# Measuring the Z Mass: Clad and RooFit Side by Side

The same measurement, twice: a binned extended maximum-likelihood fit of the Z peak on
real CMS open data, first **by hand** -- a C++ likelihood differentiated by
[**Clad**](https://github.com/vgvassilev/clad) (the automatic-differentiation plugin
bundled with ROOT) and minimized with `iminuit` -- and then with
[**RooFit**](https://root.cern/manual/roofit/), using its **codegen** backend, which
generates C++ for the whole likelihood and differentiates it with the *same* Clad. If you
already know RooFit, Part 1 shows exactly what the framework does for you under the hood;
the comparison at the end checks that both routes give the same numbers.

This is a standalone companion to
[`AutomaticDifferentiation.ipynb`](AutomaticDifferentiation.ipynb), which introduces Clad
from scratch and fits the same data with a Breit-Wigner line shape.

## Setup

**Prerequisites.** ROOT >= 6.40 (Clad is bundled and enabled by default; the Clad shipped
with older ROOT fails to generate the Hessian used below) plus `numpy`, `matplotlib`, and
`iminuit`:

```
mamba install -c conda-forge "root>=6.40" numpy matplotlib iminuit
```

Importing `ROOT` in a Jupyter kernel registers the `%%cpp` cell magic used throughout.

> **If you hit `error: redefinition of '<name>'`:** Cling does not allow redefining a
> symbol, so re-running a `%%cpp --declare` cell fails. Recovery: **Kernel -> Restart**,
> then re-run the cells above yours.


In [ ]:
import numpy as np
import ROOT
from iminuit import Minuit
import matplotlib.pyplot as plt

# warm up the Cling + Clad JIT (a few seconds) so later cells are snappy
ROOT.gInterpreter.Declare("#include <Math/CladDerivator.h>")
ROOT.gInterpreter.Declare("double _warm(double z){ return z*z; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(_warm);")
print("ROOT", ROOT.gROOT.GetVersion())

## The measurement

We model the dimuon spectrum as a **Gaussian peak** on a falling **exponential
background**, both normalized *analytically* over the fit window $[70, 110]$ GeV.
The reason why we don't use the Breit Wigner here is so simplify the normalization integrals that we have to implement by hand.

$$S(m) = \frac{e^{-(m-\mu)^2/(2\sigma^2)}}{I_S(\mu,\sigma)}, \qquad
  B(m) = \frac{e^{c\,m}}{I_B(c)},$$

$$I_S = \sqrt{\tfrac{\pi}{2}}\,\sigma\left[\operatorname{erf}\Bigl(\tfrac{110-\mu}{\sqrt{2}\,\sigma}\Bigr)
       - \operatorname{erf}\Bigl(\tfrac{70-\mu}{\sqrt{2}\,\sigma}\Bigr)\right], \qquad
  I_B = \frac{e^{110\,c} - e^{70\,c}}{c}.$$

(The natural Z line shape is a Breit-Wigner, but detector resolution smears it toward a
Gaussian anyway, and the Gaussian has an analytic normalization -- which makes this model
*exactly* expressible in RooFit with two standard pdf classes, `RooGaussian` and
`RooExponential`. The full notebook fits the Breit-Wigner instead.)

The binned **extended-Poisson** likelihood evaluates the density at the bin centers $m_i$
(bin width $w$ = 1 GeV):

$$\nu_i = w\left[N_s\,S(m_i) + N_b\,B(m_i)\right], \qquad
  \mathrm{NLL} = \sum_i \bigl(\nu_i - n_i \ln \nu_i\bigr).$$

Evaluating at bin centers is exactly what RooFit does with binned data by default, so
Parts 1 and 2 compute the *same* likelihood. **Note**: the normalizations depend on the
shape parameters, so $\partial(\mathrm{NLL})/\partial\mu$ has a term flowing through
$\operatorname{erf}$ -- the kind of term hand-coded gradients routinely forget and Clad
does not (it knows $\operatorname{erf}' (x) = \tfrac{2}{\sqrt{\pi}}e^{-x^2}$).

The data: 40 bins of 1 GeV over 70-110 GeV of opposite-sign dimuon invariant masses from
the CMS Run2011A DoubleMu dataset ([CERN Open Data record 545](https://opendata.cern.ch/record/545)), pre-binned so there is no
data loading.


In [ ]:
COUNTS = np.array([
    34, 30, 42, 30, 35, 53, 42, 43, 42, 47, 62, 59, 70, 94, 96, 128,
    171, 263, 422, 675, 806, 873, 595, 314, 228, 107, 80, 54, 46, 31,
    27, 21, 12, 14, 13, 19, 11, 7, 13, 7], dtype=float)
M_LO, M_HI, N_BINS = 70.0, 110.0, 40
BIN_W = (M_HI - M_LO) / N_BINS
CENTERS = M_LO + (np.arange(N_BINS) + 0.5) * BIN_W
PDG_MZ = 91.1876

print(f"{int(COUNTS.sum())} events; tallest bin {CENTERS[COUNTS.argmax()]:.1f} GeV with {int(COUNTS.max())} events")
plt.figure(figsize=(6, 4))
plt.bar(CENTERS, COUNTS, width=BIN_W*0.9, color="steelblue")
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title("CMS open data: dimuon spectrum"); plt.show()

### Where the counts come from (reproducing them)

`COUNTS` is real CMS data, pre-binned so the notebook needs no network or file access: the
**CMS Run2011A DoubleMu** dataset on the [CERN Open Data Portal, record 545](https://opendata.cern.ch/record/545),
whose `Dimuon_DoubleMu.csv` carries a precomputed dimuon invariant-mass column `M`
(475k events). The counts were produced once with:

```python
import numpy as np

# Dimuon_DoubleMu.csv from record 545, e.g. via XRootD:
#   root://eospublic.cern.ch//eos/opendata/cms/Run2011A/DoubleMu/CSV/12Oct2013-v1/Dimuon_DoubleMu.csv

d = np.genfromtxt("Dimuon_DoubleMu.csv", delimiter=",", names=True)
opp = d["Q1"] * d["Q2"] < 0                          # opposite-sign muons only
COUNTS, edges = np.histogram(d["M"][opp], bins=40, range=(70, 110))
```

Change `bins`/`range` and everything downstream (fit, Hessian errors) adapts.


## Part 1: by hand, with Clad + Minuit

The parameters `[N_s, N_b, mu, sigma, c]` are packed into a `double*`, and the bin counts
come in as a `const double*`. The model is written with small unit-normalized pdf
functions, and an NLL that calls them in a loop. Clad differentiates through **nested
function calls** (and through `std::erf`) without any special treatment. From the one NLL
we generate both the gradient (for the fit) and the Hessian (for the error bars).


In [ ]:
%%cpp --declare

const int NBINS = 40;
const double MLO = 70.0;
const double MHI = 110.0;
// unit-normalized pdfs over [MLO, MHI]; the normalization integrals are analytic
double gaussian_pdf(double m, double mu, double sigma)
{
   const double s2 = std::sqrt(2.0) * sigma;
   double norm = std::sqrt(M_PI / 2.0) * sigma *
                 (std::erf((MHI - mu) / s2) - std::erf((MLO - mu) / s2));
   return std::exp(-0.5 * (m - mu) * (m - mu) / (sigma * sigma)) / norm;
}
double exponential_pdf(double m, double c)
{
   double norm = (std::exp(c * MHI) - std::exp(c * MLO)) / c;
   return std::exp(c * m) / norm;
}
double nll(double *p, double const *n)
{
   const double binw = (MHI - MLO) / NBINS;
   double Ns = p[0], Nb = p[1], mu = p[2], sigma = p[3], c = p[4];
   double val = 0.0;
   for (int i = 0; i < NBINS; ++i) {
      double m = MLO + (i + 0.5) * binw;
      double nu = binw * (Ns * gaussian_pdf(m, mu, sigma) + Nb * exponential_pdf(m, c));
      val += nu - n[i] * std::log(nu);
   }
   return val;
}

**Naming of the generated functions.** Reverse mode is `<f>_grad`, the Hessian
`<f>_hessian`, forward mode `<f>_darg<k>` (one per input `k`). One wrinkle: when the
differentiated argument is an **array** *and* the function takes further arguments (as
`nll` does, taking the parameters **and** the bin counts), Clad appends an index:
`nll_grad_0` and `nll_hessian_0`. Plain-scalar functions and a lone array argument keep
the bare name.


In [ ]:
%%cpp
clad::gradient(nll, "p");     // -> nll_grad_0
clad::hessian(nll, "p[0:4]"); // -> nll_hessian_0

### The fit with Minuit driven by the Clad gradient

First spot-check the fresh gradient against a central finite difference (AD differentiates
the code you wrote, not the math you meant -- always worth one cheap check), then fit. The
C++ NLL is the cost; the Clad gradient is the Jacobian.


In [ ]:
# spot-check the fresh gradient against finite differences
p0 = np.array([4300.0, 1400.0, 91.0, 3.0, -0.03])
ga = np.zeros(5); ROOT.nll_grad_0(p0, COUNTS, ga)
gn = np.zeros(5)
for i in range(5):
    h = 1e-6 * max(1.0, abs(p0[i]))
    pp = p0.copy(); pp[i] += h
    pm = p0.copy(); pm[i] -= h
    gn[i] = (ROOT.nll(pp, COUNTS) - ROOT.nll(pm, COUNTS)) / (2*h)
print("gradient check vs finite differences:", "PASS" if np.allclose(ga, gn, rtol=1e-3) else "FAIL")

In [ ]:
def cost(par):
    return ROOT.nll(np.ascontiguousarray(par, dtype=float), COUNTS)
def grad(par):
    d = np.zeros(5)
    ROOT.nll_grad_0(np.ascontiguousarray(par, dtype=float), COUNTS, d); return d

m = Minuit(cost, p0, grad=grad, name=["N_s", "N_b", "mu", "sigma", "c"])
m.errordef = Minuit.LIKELIHOOD    # plain NLL: 1-sigma at delta-NLL = 0.5
m.limits = [(1, None), (1, None), (85, 97), (0.5, 12), (-0.2, -1e-3)]
m.migrad()
fit = np.array(m.values); Ns, Nb, mu_hat, sigma_hat, c_hat = fit
print(f"valid minimum: {m.valid}   ({m.nfcn} cost calls, {m.ngrad} gradient calls, NLL = {m.fval:.2f})")
print(f"  N_s = {Ns:8.1f}   N_b = {Nb:8.1f}   (sum {Ns+Nb:.0f}, data {COUNTS.sum():.0f})")
print(f"  mu = {mu_hat:.4f} GeV   sigma = {sigma_hat:.4f} GeV   c = {c_hat:.5f} /GeV")

### Uncertainties by inverting the AD Hessian

For a plain NLL the covariance is exactly the inverse Hessian at the minimum, $C = H^{-1}$;
1-sigma errors are the square roots of its diagonal. `clad::hessian` generated
`nll_hessian_0` from the same source as the gradient -- and we cross-check against
Minuit's numeric `m.hesse()`.


In [ ]:
H = np.zeros(25)
ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
cov = np.linalg.inv(H.reshape(5, 5))
err = np.sqrt(np.diag(cov))
m.hesse()   # Minuit's numeric Hesse, as an independent cross-check

print(f"{'param':6} {'value':>12} {'Clad Hessian':>14} {'Minuit Hesse':>14}")
for name, vv, ee, me in zip(["N_s", "N_b", "mu", "sigma", "c"], fit, err, np.array(m.errors)):
    print(f"{name:6} {vv:12.4f} {ee:14.4f} {me:14.4f}")
print()
print(f"RESULT:  mu = {mu_hat:.3f} +/- {err[2]:.3f} GeV (stat.)   PDG m_Z: {PDG_MZ:.3f} GeV")
print(f"         N_s = {Ns:.0f} +/- {err[0]:.0f} reconstructed Z -> mu mu decays")

### The fitted curve

(The Gaussian peak position sits a few hundred MeV below the PDG $m_Z$: final-state
radiation and the asymmetric Breit-Wigner tail drag the observed peak down, and a
symmetric Gaussian dutifully reports that. For this notebook the physics bias is beside
the point -- what matters is that Part 2 must land on the *same* number.)


In [ ]:
from math import erf, sqrt, pi

def curves(par):
    Ns, Nb, mu, sigma, c = par
    s2 = sqrt(2.0) * sigma
    norm_s = sqrt(pi / 2.0) * sigma * (erf((M_HI - mu) / s2) - erf((M_LO - mu) / s2))
    S = np.exp(-0.5 * ((CENTERS - mu) / sigma) ** 2) / norm_s
    B = np.exp(c * CENTERS) / ((np.exp(c * M_HI) - np.exp(c * M_LO)) / c)
    return BIN_W * Ns * S, BIN_W * Nb * B

sig, bkg = curves(fit)
plt.figure(figsize=(7.5, 5))
plt.errorbar(CENTERS, COUNTS, yerr=np.sqrt(COUNTS), fmt="o", ms=4, color="black", label="CMS open data", zorder=5)
plt.plot(CENTERS, sig+bkg, "-", color="crimson", lw=2, label=f"Gaussian fit (mu={mu_hat:.2f})")
plt.plot(CENTERS, bkg, ":", color="steelblue", label="background")
plt.axvline(PDG_MZ, color="gray", ls=":", alpha=0.8)
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title(f"Z peak from CMS open data: mu = {mu_hat:.2f} +/- {err[2]:.2f} GeV (stat.)")
plt.legend(); plt.show()

## Part 2: the same fit in RooFit

Now the framework version. The mapping is one-to-one:

| by hand (Part 1) | RooFit |
|---|---|
| `double *p` parameters | a `RooRealVar` for each parameter |
| `gaussian_pdf(m, mu, sigma)`, erf normalization | `RooGaussian` (same analytic integral built in) |
| `exponential_pdf(m, c)` | `RooExponential` |
| $\nu_i = w\,(N_s S_i + N_b B_i)$ | extended `RooAddPdf` with yield coefficients |
| `COUNTS` array | `RooDataHist` |
| extended Poisson NLL loop | `fitTo(..., Extended=True)` |
| `clad::gradient(nll, "p")` | `EvalBackend="codegen"` -- Clad again |
| `clad::hessian` $\to C = H^{-1}$ | HESSE |

`RooGaussian` and `RooExponential` normalize themselves with the *same* closed-form
integrals we coded in Part 1, and for binned data RooFit by default evaluates the pdf at
the bin centers -- so the likelihood below is, term for term, the one from Part 1.

The punchline is the evaluation backend. With `EvalBackend="codegen"`, RooFit does not
interpret the model graph at fit time: it **generates C++ source for the entire NLL,
compiles it with Cling, and hands it to Clad** for the gradient that drives Minuit --
literally the Part 1 workflow, automated. Watch for the `Function JIT time` and
`Gradient generation time` INFO lines when the fit starts.


In [ ]:
mass = ROOT.RooRealVar("mass", "dimuon invariant mass", M_LO, M_HI, "GeV")
mass.setBins(N_BINS)

mu_rf    = ROOT.RooRealVar("mu", "peak position", 91.0, 85.0, 97.0, "GeV")
sigma_rf = ROOT.RooRealVar("sigma", "peak width", 3.0, 0.5, 12.0, "GeV")
c_rf     = ROOT.RooRealVar("c", "background slope", -0.03, -0.2, -1e-3)

# same line shapes as the C++ in Part 1, including their analytic normalization
sig_pdf = ROOT.RooGaussian("sig", "Gaussian peak", mass, mu_rf, sigma_rf)
bkg_pdf = ROOT.RooExponential("bkg", "exponential background", mass, c_rf)

Ns_rf = ROOT.RooRealVar("Ns", "signal yield", 4300.0, 1.0, 1e6)
Nb_rf = ROOT.RooRealVar("Nb", "background yield", 1400.0, 1.0, 1e6)
model = ROOT.RooAddPdf("model", "sig + bkg", [sig_pdf, bkg_pdf], [Ns_rf, Nb_rf])

# the binned data: fill a TH1 with COUNTS and import it
th1 = ROOT.TH1D("h_dimuon", "dimuon mass", N_BINS, M_LO, M_HI)
for i, cnt in enumerate(COUNTS):
    th1.SetBinContent(i + 1, cnt)
data = ROOT.RooDataHist("data", "binned dimuon data", [mass], Import=th1)
print("model and data ready:", int(data.sumEntries()), "events")

In [ ]:
nll = model.createNLL(data,
                     Extended=True,            # Poisson term for the total yield
                     EvalBackend="codegen",    # generate C++ for the NLL, differentiate it with Clad
                     )

minim = ROOT.RooMinimizer(nll)
minim.setStrategy(0) # fastest Minuit 2 strategy
minim.setPrintLevel(-1)
minim.minimize("Minuit2", "migrad")
result = minim.save()

result.Print()

## Inspeciting the generated code

There are also experimental RooFit features to inspect and debug the generated code, if the likelihood was created with the `"codegen"` backend!

In [ ]:
ROOT.RooFit.Experimental.writeCodegenDebugMacro(nll, "debug")

In [ ]:
!head -68 debug.C | tail -62

### Side by side

Same likelihood, same minimizer family, two very different amounts of user-written code.
The central values and the uncertainties (Clad Hessian vs. HESSE on the codegen NLL)
should agree to the minimizer's convergence tolerance -- pulls of a few percent of a
sigma at most.


In [ ]:
roofit_vals = {v.GetName(): (v.getVal(), v.getError()) for v in [Ns_rf, Nb_rf, mu_rf, sigma_rf, c_rf]}
pairs = [("N_s", "Ns"), ("N_b", "Nb"), ("mu", "mu"), ("sigma", "sigma"), ("c", "c")]

print(f"{'param':7} {'Clad+iminuit':>24} {'RooFit codegen':>24} {'diff/sigma':>11}")
for i, (nm, rf) in enumerate(pairs):
    v_rf, e_rf = roofit_vals[rf]
    pull = (fit[i] - v_rf) / err[i]
    print(f"{nm:7} {fit[i]:12.5f} +/- {err[i]:8.5f} {v_rf:12.5f} +/- {e_rf:8.5f} {pull:11.3f}")

print(f"\npeak:  Clad+iminuit {fit[2]:.3f} +/- {err[2]:.3f} GeV   "
      f"RooFit {roofit_vals['mu'][0]:.3f} +/- {roofit_vals['mu'][1]:.3f} GeV   PDG m_Z {PDG_MZ:.3f} GeV")

In [ ]:
rf_fit = np.array([roofit_vals[k][0] for _, k in pairs])
sig_ad, bkg_ad = curves(fit)
sig_rf, bkg_rf = curves(rf_fit)

plt.figure(figsize=(7.5, 5))
plt.errorbar(CENTERS, COUNTS, yerr=np.sqrt(COUNTS), fmt="o", ms=4, color="black",
             label="CMS open data", zorder=5)
plt.plot(CENTERS, sig_ad + bkg_ad, "-", color="crimson", lw=2,
         label=f"Clad + iminuit (mu = {fit[2]:.3f})")
plt.plot(CENTERS, sig_rf + bkg_rf, "--", color="darkorange", lw=2,
         label=f"RooFit codegen (mu = {rf_fit[2]:.3f})")
plt.axvline(PDG_MZ, color="gray", ls=":", alpha=0.8)
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title("Z peak from CMS open data: two fits, one answer")
plt.legend(); plt.show()

## Conclusions

Both routes compute the same likelihood and land on the same answer:

- **By hand**, every ingredient is explicit: you write the pdfs and their analytic
  normalizations, Clad generates the exact gradient and Hessian from the C++ source,
  Minuit minimizes, and $C = H^{-1}$ gives the errors. About thirty lines, nothing hidden.
- **RooFit** assembles the identical NLL from a model description -- `RooGaussian`,
  `RooExponential`, an extended `RooAddPdf`, a `RooDataHist` -- and with
  `EvalBackend="codegen"` it then does *exactly what Part 1 did*: emit C++ for the NLL and
  let Clad differentiate it.

So the by-hand exercise is not an alternative to RooFit; it is a transparent view of what
RooFit's AD-based likelihood evaluation does behind `fitTo`. When your model outgrows a
Gaussian plus an exponential -- convolutions, per-event terms, hundreds of nuisance
parameters -- the framework writes and differentiates the code you would not want to
maintain by hand.
